# Multimodal Medical Summarization — Replacement Notebook

**Project goal:** generate an English medical summary from a code-mixed medical question and its associated medical image.

This notebook is organized into 7 sections:

1. Environment and configuration
2. Dataset loading and validation
3. Leakage-safe image-level train/validation/test split
4. Text-only BART baseline
5. Vision Transformer (ViT) image encoder
6. Multimodal ViT + BART model
7. Training, generation and evaluation

**Important:** This is an academic summarization prototype, not a diagnostic system. Generated text must be reviewed by a qualified human before any clinical use.


## Section 1 — Environment, imports and configuration

In [ ]:
# If needed, run this once in a notebook cell:
# %pip install -U torch torchvision transformers accelerate evaluate rouge_score pandas pyarrow scikit-learn pillow matplotlib tqdm

import os
import sys
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    BartForConditionalGeneration,
    ViTImageProcessor,
    ViTModel,
)
from transformers.modeling_outputs import BaseModelOutput

warnings.filterwarnings("ignore")

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# -----------------------------
# Project configuration
# -----------------------------
TEXT_COLUMN = "Codemixed_Question"
TARGET_COLUMN = "summary"
IMAGE_COLUMN = "Image_path"

BART_NAME = "facebook/bart-base"
VIT_NAME = "google/vit-base-patch16-224-in21k"

MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 128
IMAGE_SIZE = 224

# RTX 3050 6GB: start conservatively.
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS = 3
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01

# Freeze ViT initially to reduce memory and training time.
FREEZE_VIT = True

OUTPUT_DIR = Path("outputs_multimodal")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")


## Section 2 — Load and validate the dataset

In [ ]:
from pathlib import Path

def find_parquet_file():
    candidates = [
        Path("train-00000-of-00001-d99c2c6aef2fc2d4.parquet"),
        Path("data/train-00000-of-00001-d99c2c6aef2fc2d4.parquet"),
        Path.home() / "Downloads/train-00000-of-00001-d99c2c6aef2fc2d4.parquet",
    ]
    for p in candidates:
        if p.exists():
            return p

    # Search Downloads for any parquet as a fallback.
    downloads = Path.home() / "Downloads"
    if downloads.exists():
        files = sorted(downloads.glob("*.parquet"))
        if files:
            return files[0]

    raise FileNotFoundError(
        "Parquet file not found. Put the dataset parquet file in the notebook folder "
        "or in ~/Downloads, then run this cell again."
    )

parquet_path = find_parquet_file()
df = pd.read_parquet(parquet_path)

print("Dataset:", parquet_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

required = {TEXT_COLUMN, TARGET_COLUMN, IMAGE_COLUMN}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Basic cleaning
df = df[[TEXT_COLUMN, TARGET_COLUMN, IMAGE_COLUMN]].copy()
df = df.dropna(subset=[TEXT_COLUMN, TARGET_COLUMN, IMAGE_COLUMN]).reset_index(drop=True)

for col in [TEXT_COLUMN, TARGET_COLUMN, IMAGE_COLUMN]:
    df[col] = df[col].astype(str).str.strip()

df = df[
    (df[TEXT_COLUMN] != "") &
    (df[TARGET_COLUMN] != "") &
    (df[IMAGE_COLUMN] != "")
].reset_index(drop=True)

print("After cleaning:", df.shape)
print("Unique images:", df[IMAGE_COLUMN].nunique())
print("Unique questions:", df[TEXT_COLUMN].nunique())
print("Unique summaries:", df[TARGET_COLUMN].nunique())

display(df.head(3))


In [ ]:
# ---------------------------------------------------------
# Locate the image root automatically.
# Image_path values look like:
# Multimodal_images/swollen tonsils/Image_2.jpg
# ---------------------------------------------------------

sample_rel = Path(df[IMAGE_COLUMN].iloc[0])

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.home(),
    Path.home() / "Documents" / "Images_dataset",
    Path.home() / "Downloads",
]

def locate_image_root(relative_path):
    for root in candidate_roots:
        candidate = root / relative_path
        if candidate.is_file():
            return root
    return None

IMAGE_ROOT = locate_image_root(sample_rel)

if IMAGE_ROOT is None:
    print("Could not automatically locate the image root.")
    print("Current directory:", Path.cwd())
    print("Example relative image:", sample_rel)
    print("\nSet IMAGE_ROOT manually, for example:")
    print('IMAGE_ROOT = Path("/your/path/to/Images_dataset")')
else:
    print("Detected IMAGE_ROOT:", IMAGE_ROOT)

def resolve_image_path(relative_path):
    p = Path(relative_path)
    if p.is_absolute() and p.is_file():
        return p
    if IMAGE_ROOT is not None:
        return IMAGE_ROOT / p
    return p

df["Image_path_full"] = df[IMAGE_COLUMN].apply(resolve_image_path)

df["image_exists"] = df["Image_path_full"].apply(lambda p: Path(p).is_file())

print("Images found:", int(df["image_exists"].sum()), "/", len(df))

missing_images = df.loc[~df["image_exists"], IMAGE_COLUMN].head(10).tolist()
if missing_images:
    print("First missing image paths:")
    for p in missing_images:
        print(" -", p)

if df["image_exists"].sum() == 0:
    raise FileNotFoundError(
        "No images were found. Set IMAGE_ROOT manually before continuing."
    )

# For a clean experiment, remove records whose image file is unavailable.
df = df[df["image_exists"]].drop(columns=["image_exists"]).reset_index(drop=True)

print("Usable records:", len(df))
print("Usable unique images:", df[IMAGE_COLUMN].nunique())


In [ ]:
# Visual sanity check
import matplotlib.pyplot as plt

n = min(3, len(df))
fig, axes = plt.subplots(n, 1, figsize=(10, 4*n))
if n == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, df.head(n).iterrows()):
    image = Image.open(row["Image_path_full"]).convert("RGB")
    ax.imshow(image)
    ax.axis("off")
    ax.set_title(
        f"Question: {row[TEXT_COLUMN][:180]}...\n"
        f"Reference: {row[TARGET_COLUMN][:180]}..."
    )

plt.tight_layout()
plt.show()


## Section 3 — Leakage-safe image-level splitting

In [ ]:
# IMPORTANT:
# The same image can have multiple questions/records.
# Therefore split UNIQUE IMAGES first, then assign all rows belonging to
# each image to the same split.

unique_images = np.array(sorted(df[IMAGE_COLUMN].unique()))

train_images, temp_images = train_test_split(
    unique_images,
    test_size=0.20,
    random_state=SEED
)

val_images, test_images = train_test_split(
    temp_images,
    test_size=0.50,
    random_state=SEED
)

train_set = set(train_images)
val_set = set(val_images)
test_set = set(test_images)

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

df["split"] = df[IMAGE_COLUMN].map(
    lambda x: "train" if x in train_set else ("validation" if x in val_set else "test")
)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("Unique image split:")
print(" Train:", len(train_images))
print(" Validation:", len(val_images))
print(" Test:", len(test_images))

print("\nRecord split:")
print(" Train:", len(train_df))
print(" Validation:", len(val_df))
print(" Test:", len(test_df))

# Final leakage checks
assert set(train_df[IMAGE_COLUMN]).isdisjoint(set(val_df[IMAGE_COLUMN]))
assert set(train_df[IMAGE_COLUMN]).isdisjoint(set(test_df[IMAGE_COLUMN]))
assert set(val_df[IMAGE_COLUMN]).isdisjoint(set(test_df[IMAGE_COLUMN]))

print("\n✓ No image leakage between splits.")


In [ ]:
# Save the exact split so future experiments use the same test set.
split_file = OUTPUT_DIR / "dataset_split.csv"
df[[TEXT_COLUMN, TARGET_COLUMN, IMAGE_COLUMN, "split"]].to_csv(split_file, index=False)

print("Saved split:", split_file)
print("\nRecords per image:")
print(df.groupby(IMAGE_COLUMN).size().describe())


## Section 4 — Text-only BART baseline

In [ ]:
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained(BART_NAME)

class TextSummaryDataset(Dataset):
    def __init__(self, frame, tokenizer):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]

        source = self.tokenizer(
            row[TEXT_COLUMN],
            max_length=MAX_SOURCE_LENGTH,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        target = self.tokenizer(
            row[TARGET_COLUMN],
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        labels = target["input_ids"].squeeze(0)
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids": source["input_ids"].squeeze(0),
            "attention_mask": source["attention_mask"].squeeze(0),
            "labels": labels,
        }

train_text_ds = TextSummaryDataset(train_df, tokenizer)
val_text_ds = TextSummaryDataset(val_df, tokenizer)
test_text_ds = TextSummaryDataset(test_df, tokenizer)

train_loader = DataLoader(train_text_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_text_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_text_ds, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


In [ ]:
bart_baseline = BartForConditionalGeneration.from_pretrained(BART_NAME)
bart_baseline.to(DEVICE)

# Gradient checkpointing reduces memory usage on smaller GPUs.
if hasattr(bart_baseline, "gradient_checkpointing_enable"):
    bart_baseline.gradient_checkpointing_enable()

optimizer = torch.optim.AdamW(
    bart_baseline.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0

    if training:
        optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(loader, leave=False)):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):
            outputs = model(**batch)
            loss = outputs.loss / (GRAD_ACCUM_STEPS if training else 1)

        if training:
            scaler.scale(loss).backward()

            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * (GRAD_ACCUM_STEPS if training else 1)

    return total_loss / max(len(loader), 1)

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    train_loss = run_epoch(bart_baseline, train_loader, optimizer)

    with torch.no_grad():
        val_loss = run_epoch(bart_baseline, val_loader)

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        bart_baseline.save_pretrained(OUTPUT_DIR / "bart_baseline")
        tokenizer.save_pretrained(OUTPUT_DIR / "bart_baseline")
        print("  ✓ Saved best BART baseline")


In [ ]:
# ROUGE evaluation for the text-only baseline
import evaluate

rouge = evaluate.load("rouge")

best_bart = BartForConditionalGeneration.from_pretrained(
    OUTPUT_DIR / "bart_baseline"
).to(DEVICE)

best_bart.eval()

predictions = []
references = []

for batch_idx, batch in enumerate(tqdm(test_loader)):
    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)

    with torch.no_grad():
        generated = best_bart.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=4,
            early_stopping=True,
        )

    predictions.extend(
        tokenizer.batch_decode(generated, skip_special_tokens=True)
    )

    labels = batch["labels"].clone()
    labels[labels == -100] = tokenizer.pad_token_id

    references.extend(
        tokenizer.batch_decode(labels, skip_special_tokens=True)
    )

baseline_scores = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

print("BART baseline:")
for k, v in baseline_scores.items():
    print(f"{k}: {v:.4f}")

pd.DataFrame({
    "prediction": predictions,
    "reference": references,
}).to_csv(OUTPUT_DIR / "bart_baseline_predictions.csv", index=False)


## Section 5 — Vision Transformer (ViT) image encoder

In [ ]:
image_processor = ViTImageProcessor.from_pretrained(VIT_NAME)
vit = ViTModel.from_pretrained(VIT_NAME)

if FREEZE_VIT:
    for param in vit.parameters():
        param.requires_grad = False

vit.to(DEVICE)
vit.eval() if FREEZE_VIT else vit.train()

print("ViT hidden size:", vit.config.hidden_size)
print("ViT image size:", vit.config.image_size)
print("ViT patch size:", vit.config.patch_size)


In [ ]:
# Test ViT on one image
sample_image = Image.open(train_df["Image_path_full"].iloc[0]).convert("RGB")
pixel_values = image_processor(
    images=sample_image,
    return_tensors="pt"
)["pixel_values"].to(DEVICE)

with torch.no_grad():
    vit_output = vit(pixel_values=pixel_values)

print("pixel_values:", tuple(pixel_values.shape))
print("ViT last_hidden_state:", tuple(vit_output.last_hidden_state.shape))
print("CLS embedding:", tuple(vit_output.last_hidden_state[:, 0].shape))


## Section 6 — Multimodal ViT + BART model

In [ ]:
class MultimodalMedicalDataset(Dataset):
    def __init__(
        self,
        frame,
        tokenizer,
        image_processor,
        max_source_length=256,
        max_target_length=128,
    ):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        self.max_source_length = max_source_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]

        text_inputs = self.tokenizer(
            row[TEXT_COLUMN],
            max_length=self.max_source_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        target_inputs = self.tokenizer(
            row[TARGET_COLUMN],
            max_length=self.max_target_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        image = Image.open(row["Image_path_full"]).convert("RGB")
        image_inputs = self.image_processor(
            images=image,
            return_tensors="pt",
        )

        labels = target_inputs["input_ids"].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": text_inputs["input_ids"].squeeze(0),
            "attention_mask": text_inputs["attention_mask"].squeeze(0),
            "pixel_values": image_inputs["pixel_values"].squeeze(0),
            "labels": labels,
        }

train_mm_ds = MultimodalMedicalDataset(
    train_df, tokenizer, image_processor,
    MAX_SOURCE_LENGTH, MAX_TARGET_LENGTH
)
val_mm_ds = MultimodalMedicalDataset(
    val_df, tokenizer, image_processor,
    MAX_SOURCE_LENGTH, MAX_TARGET_LENGTH
)
test_mm_ds = MultimodalMedicalDataset(
    test_df, tokenizer, image_processor,
    MAX_SOURCE_LENGTH, MAX_TARGET_LENGTH
)

train_mm_loader = DataLoader(train_mm_ds, batch_size=BATCH_SIZE, shuffle=True)
val_mm_loader = DataLoader(val_mm_ds, batch_size=BATCH_SIZE, shuffle=False)
test_mm_loader = DataLoader(test_mm_ds, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_mm_loader))
print({k: tuple(v.shape) for k, v in batch.items()})


In [ ]:
class ViTBartMultimodal(nn.Module):
    """
    Multimodal encoder-decoder:
      text -> BART encoder
      image -> ViT
      ViT features -> projection
      text + image tokens -> concatenated encoder sequence
      fused sequence -> BART decoder
    """

    def __init__(
        self,
        bart_name=BART_NAME,
        vit_name=VIT_NAME,
        freeze_vit=True,
    ):
        super().__init__()

        self.bart = BartForConditionalGeneration.from_pretrained(bart_name)
        self.vit = ViTModel.from_pretrained(vit_name)

        bart_hidden = self.bart.config.d_model
        vit_hidden = self.vit.config.hidden_size

        self.image_projection = nn.Sequential(
            nn.Linear(vit_hidden, bart_hidden),
            nn.LayerNorm(bart_hidden),
            nn.GELU(),
            nn.Dropout(0.1),
        )

        self.fusion_layer = nn.Sequential(
            nn.Linear(bart_hidden, bart_hidden),
            nn.LayerNorm(bart_hidden),
            nn.GELU(),
            nn.Dropout(0.1),
        )

        self.freeze_vit = freeze_vit

        if freeze_vit:
            for p in self.vit.parameters():
                p.requires_grad = False

        if hasattr(self.bart, "gradient_checkpointing_enable"):
            self.bart.gradient_checkpointing_enable()

    def encode_multimodal(
        self,
        input_ids,
        attention_mask,
        pixel_values,
    ):
        # Text branch
        text_encoder_outputs = self.bart.model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        text_features = text_encoder_outputs.last_hidden_state

        # Image branch
        if self.freeze_vit:
            with torch.no_grad():
                image_outputs = self.vit(
                    pixel_values=pixel_values,
                    return_dict=True,
                )
        else:
            image_outputs = self.vit(
                pixel_values=pixel_values,
                return_dict=True,
            )

        # Use all ViT patch tokens, not only CLS.
        image_features = self.image_projection(
            image_outputs.last_hidden_state
        )

        # Light fusion transformation.
        image_features = self.fusion_layer(image_features)

        # Concatenate text tokens and image tokens.
        fused_features = torch.cat(
            [text_features, image_features],
            dim=1,
        )

        image_attention = torch.ones(
            image_features.size(0),
            image_features.size(1),
            dtype=attention_mask.dtype,
            device=attention_mask.device,
        )

        fused_attention_mask = torch.cat(
            [attention_mask, image_attention],
            dim=1,
        )

        return fused_features, fused_attention_mask

    def forward(
        self,
        input_ids,
        attention_mask,
        pixel_values,
        labels=None,
    ):
        fused_features, fused_attention_mask = self.encode_multimodal(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
        )

        encoder_outputs = BaseModelOutput(
            last_hidden_state=fused_features
        )

        return self.bart(
            input_ids=None,
            attention_mask=fused_attention_mask,
            encoder_outputs=encoder_outputs,
            labels=labels,
            return_dict=True,
        )

    @torch.no_grad()
    def generate_summary(
        self,
        input_ids,
        attention_mask,
        pixel_values,
        **generation_kwargs,
    ):
        fused_features, fused_attention_mask = self.encode_multimodal(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
        )

        encoder_outputs = BaseModelOutput(
            last_hidden_state=fused_features
        )

        return self.bart.generate(
            input_ids=input_ids,
            attention_mask=fused_attention_mask,
            encoder_outputs=encoder_outputs,
            **generation_kwargs,
        )

model = ViTBartMultimodal(
    bart_name=BART_NAME,
    vit_name=VIT_NAME,
    freeze_vit=FREEZE_VIT,
).to(DEVICE)

print("Multimodal model created.")
print("BART hidden size:", model.bart.config.d_model)
print("ViT hidden size:", model.vit.config.hidden_size)


In [ ]:
# Quick forward-pass test before starting a long training run.
model.train()

test_batch = next(iter(train_mm_loader))
test_batch = {k: v.to(DEVICE) for k, v in test_batch.items()}

with torch.amp.autocast(
    device_type="cuda",
    enabled=torch.cuda.is_available()
):
    test_outputs = model(**test_batch)

print("Forward pass successful.")
print("Loss:", float(test_outputs.loss.detach().cpu()))


In [ ]:
# Train the multimodal model
mm_optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

mm_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

def run_mm_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    if FREEZE_VIT:
        model.vit.eval()

    total_loss = 0.0

    if training:
        optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(loader, leave=False)):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):
            outputs = model(**batch)
            loss = outputs.loss / (GRAD_ACCUM_STEPS if training else 1)

        if training:
            mm_scaler.scale(loss).backward()

            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(loader):
                mm_scaler.step(optimizer)
                mm_scaler.update()
                optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * (GRAD_ACCUM_STEPS if training else 1)

    return total_loss / max(len(loader), 1)

best_mm_val = float("inf")
best_mm_path = OUTPUT_DIR / "vit_bart_multimodal"

for epoch in range(NUM_EPOCHS):
    train_loss = run_mm_epoch(model, train_mm_loader, mm_optimizer)

    with torch.no_grad():
        val_loss = run_mm_epoch(model, val_mm_loader)

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_mm_val:
        best_mm_val = val_loss
        best_mm_path.mkdir(parents=True, exist_ok=True)
        torch.save(
            model.state_dict(),
            best_mm_path / "model.pt"
        )
        tokenizer.save_pretrained(best_mm_path)
        image_processor.save_pretrained(best_mm_path)
        print("  ✓ Saved best multimodal checkpoint")


## Section 7 — Test generation, ROUGE evaluation and qualitative comparison

In [ ]:
# Reload best multimodal checkpoint
best_model = ViTBartMultimodal(
    bart_name=BART_NAME,
    vit_name=VIT_NAME,
    freeze_vit=FREEZE_VIT,
).to(DEVICE)

best_model.load_state_dict(
    torch.load(
        OUTPUT_DIR / "vit_bart_multimodal" / "model.pt",
        map_location=DEVICE
    )
)
best_model.eval()

mm_predictions = []
mm_references = []
mm_questions = []
mm_images = []

for batch_idx, batch in enumerate(tqdm(test_mm_loader)):
    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    pixel_values = batch["pixel_values"].to(DEVICE)

    generated = best_model.generate_summary(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values,
        max_new_tokens=MAX_TARGET_LENGTH,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=3,
    )

    mm_predictions.extend(
        tokenizer.batch_decode(
            generated,
            skip_special_tokens=True
        )
    )

    labels = batch["labels"].clone()
    labels[labels == -100] = tokenizer.pad_token_id

    mm_references.extend(
        tokenizer.batch_decode(
            labels,
            skip_special_tokens=True
        )
    )

# Preserve row-level metadata in the same test order.
test_meta = test_df.reset_index(drop=True)

results_df = pd.DataFrame({
    "question": test_meta[TEXT_COLUMN].tolist(),
    "image_path": test_meta[IMAGE_COLUMN].tolist(),
    "reference": mm_references,
    "prediction": mm_predictions,
})

mm_scores = rouge.compute(
    predictions=mm_predictions,
    references=mm_references,
    use_stemmer=True,
)

print("Multimodal ViT + BART:")
for k, v in mm_scores.items():
    print(f"{k}: {v:.4f}")

results_df.to_csv(
    OUTPUT_DIR / "vit_bart_multimodal_predictions.csv",
    index=False
)


In [ ]:
# Compare baseline and multimodal metrics.
comparison = pd.DataFrame([
    {
        "model": "BART text-only baseline",
        "input": "Text",
        "rouge1": baseline_scores.get("rouge1", np.nan),
        "rouge2": baseline_scores.get("rouge2", np.nan),
        "rougeL": baseline_scores.get("rougeL", np.nan),
    },
    {
        "model": "ViT + BART multimodal",
        "input": "Text + Image",
        "rouge1": mm_scores.get("rouge1", np.nan),
        "rouge2": mm_scores.get("rouge2", np.nan),
        "rougeL": mm_scores.get("rougeL", np.nan),
    },
])

display(comparison)
comparison.to_csv(
    OUTPUT_DIR / "model_comparison.csv",
    index=False
)


In [ ]:
# Show qualitative examples
pd.set_option("display.max_colwidth", 500)

display(
    results_df[
        ["question", "image_path", "reference", "prediction"]
    ].head(10)
)


In [ ]:
# Optional: BERTScore
# Install if required:
# %pip install -U bert-score

RUN_BERTSCORE = False

if RUN_BERTSCORE:
    from bert_score import score as bertscore

    P, R, F1 = bertscore(
        mm_predictions,
        mm_references,
        lang="en",
        verbose=True,
    )

    print("BERTScore Precision:", float(P.mean()))
    print("BERTScore Recall:", float(R.mean()))
    print("BERTScore F1:", float(F1.mean()))

    comparison.loc[
        comparison["model"] == "ViT + BART multimodal",
        "bertscore_f1"
    ] = float(F1.mean())

    comparison.to_csv(
        OUTPUT_DIR / "model_comparison.csv",
        index=False
    )


# Final checklist

Before submitting the college project, confirm:

- [ ] Dataset contains text, image path and reference summary.
- [ ] Missing images have been checked.
- [ ] Train/validation/test are split by **unique image**, not random rows.
- [ ] No image occurs in more than one split.
- [ ] BART text-only baseline is reproducible.
- [ ] ViT is actually used during multimodal training.
- [ ] Image features are projected and fused with text features.
- [ ] The decoder receives the fused representation.
- [ ] ROUGE is calculated on the unseen image-level test set.
- [ ] At least 10 qualitative examples are included in the report.
- [ ] The report clearly states that this is a summarization prototype, not a diagnostic system.
